# 从手搓到框架，FastAPI 登场

> 认识几个主流后端框架，以及 FastAPI 和 uvicorn 各自负责什么。然后把手搓的接口重写一遍，再顺手加一个 POST。

## 那些杂活，每个后端都一样

上一节我们手搓了一个 API，也切身体会到它并不轻松。为了把一段 JSON 发出去，我们手写了路由判断、状态码、响应头、`dumps`、`encode`、404 兜底……而这才一个接口，还只是 GET。

今天开始学习后端框架。HTTP 的这些事情，用了框架就会轻松很多。

框架这个词我们已经不陌生了。讲 React 时我们说过：**框架，就是管某一摊事的一套规则**，React 管的是“UI 组件”这一摊。后端框架管的是另一摊：

- 接住请求；
- 解析请求内容；
- 把响应发回去。

我们只需要按照框架的规则，填上真正关注的部分：**“这个路径，该返回什么数据？”**

## 认识几个主流后端框架

Python 的后端框架不止一个：

| 框架 | 特点 |
| --- | --- |
| **Flask** | 老牌、轻量，长期的入门经典，生态成熟 |
| **Django** | “大而全”，自带后台管理界面、用户系统和 ORM，适合直接开发大型网站 |
| **FastAPI** | 专为写 API 而生，样板代码少，自动生成接口文档，还能根据类型声明自动解析和校验 |

这门课选择 **FastAPI**，主要有三个理由：

1. 代码少。
2. 类型校验的反馈直接。
3. 自动生成接口文档。

对于以 API 为主、希望快速获得这些能力的 Python 新项目，FastAPI 是很合适的选择。

FastAPI 官网的 [User Guide](https://fastapi.tiangolo.com/tutorial/) 写得很好，不只讲“是什么、怎么用”，还常常解释“为什么”。官网的 [About](https://fastapi.tiangolo.com/alternatives/) 页面还专门比较了 Flask、Django 等框架，值得作为进一步学习资料。

> 和 4.3 选择 React 时一样：框架之间的概念是相通的。把 FastAPI 用明白以后，再看 Flask、Django，很多概念都会变成熟悉的面孔。

## 两个角色：FastAPI 和 uvicorn

上一节的 `main.py` 同时做了两类工作：

1. 判断路径、组织响应，决定“收到这个请求后返回什么”。
2. 用 `HTTPServer(...)` 和 `serve_forever()` 守住 8000 端口，一直等待请求。

使用 FastAPI 后，这两类工作分工给两个工具：

| 5.3 手搓版的职责 | 现在由谁负责 |
| --- | --- |
| `if self.path == ...`、组织响应 | **FastAPI**：负责定义接口 |
| `HTTPServer(...)`、`serve_forever()` | **uvicorn**：负责运行服务器、监听端口 |

所以：

> **FastAPI 负责“接口该做什么”，uvicorn 负责“让接口跑起来”。**

FastAPI 自己不会守着端口等请求；uvicorn 收到请求后，会把请求交给 FastAPI 处理。

后面先亲手指挥一次 uvicorn，把它的命令认清楚；再换成官网的快捷命令 `fastapi dev`。这样以后在其他教程或报错里遇到 uvicorn，也能知道它负责哪一部分。

## 把 FastAPI 装进来

FastAPI 是第三方包。5.2 已经完整走过“安装第三方包”这套流程，这次只是把包名换成 FastAPI。

先确认自己处在项目虚拟环境中。安装前先看终端提示符，应该能看到类似 `(zero-to-tech)` 或 `(.venv)`：

~~~bash
cd ~/zero-to-tech/backend
source .venv/bin/activate
pip install "fastapi[standard]"
~~~

Windows PowerShell 激活虚拟环境：

~~~powershell
cd $HOME\zero-to-tech\backend
.venv\Scripts\Activate.ps1
pip install "fastapi[standard]"
~~~

如果项目已经使用 uv，也可以在项目目录执行：

~~~bash
uv add "fastapi[standard]"
~~~

`fastapi[standard]` 中的方括号是 pip 的“套餐”写法，表示 FastAPI 本体加上官方推荐的一套标准配件。

安装后查看依赖：

~~~bash
pip list
~~~

列表会比只安装 `requests` 时长很多。uvicorn 会随这个套餐安装进来；还会看到 `starlette`、`pydantic`、`fastapi-cli` 等依赖。一个包依赖其他包，这是正常现象。

## 用 FastAPI 重写 `/api/profile`

先把上一节的手搓版留作纪念。在 `backend` 目录中执行：

~~~bash
mv main.py handmade.py
~~~

Windows PowerShell 可以使用：

~~~powershell
Move-Item main.py handmade.py
~~~

然后新建 `main.py`，写入 FastAPI 版本：

~~~python
from fastapi import FastAPI

app = FastAPI()

profile = {
    "heroTitle": "关于我",
    "heroSubtitle": "项目，创意，灵感，心得，我的作品",
}


@app.get("/api/profile")
def get_profile():
    return profile
~~~

没了，这就是全部。

`@app.get("/api/profile")` 是一个装饰器。现在不用深入学习装饰器的语法，先读懂它的意思：

> `/api/profile` 这个路径的 GET 请求，交给下面的 `get_profile()` 函数处理。

函数返回的 Python 字典会被 FastAPI 自动转换成 JSON 响应。

### 手搓版和 FastAPI 版对照

| 手搓版（5.3 亲手写的） | FastAPI 版 |
| --- | --- |
| `if self.path == ...` 路由判断 | `@app.get(...)` 一行装饰器 |
| `send_response(200)` | 自动 |
| `send_header("Content-Type", ...)` | 自动设置 JSON 响应类型 |
| `json.dumps(...).encode(...)` | 自动把返回的 Python 字典序列化为 JSON |
| `else` 兜底 404 | 自动，没定义的路径会自动返回 404 |

FastAPI 并没有取消 HTTP 规范，而是把这些通用细节替我们完成了。我们仍然是在返回状态码、响应头和响应体，只是不用每次手写这些底层步骤。

## 跑起来：先用 uvicorn“手动挡”

启动前，确认 5.3 的手搓服务已经用 `Ctrl + C` 停掉。**同一个端口，在同一时间只能由一个程序监听。**

如果忘记停掉旧服务，启动时可能看到：

~~~text
[Errno 98] Address already in use
~~~

这表示地址或端口已经被其他程序占用。

在 `backend` 目录中直接运行 uvicorn：

~~~bash
uvicorn main:app --reload
~~~

`main:app` 可以拆开看：

~~~text
main : app
文件   变量
~~~

它的意思是：让 uvicorn 找到 `main.py` 文件里的 `app` 对象，这里不写 `.py`。

`--reload` 表示开发模式下，代码修改后自动重启。前端的 `npm run dev` 也有类似体验，后端开发时就不用每次修改后手动 `Ctrl + C` 再启动。

服务启动后，uvicorn 守着 8000 端口，并把收到的请求交给 `main.py` 里的 `app`。

## 换官网的快捷命令：`fastapi dev`

先按 `Ctrl + C` 停掉 uvicorn，再用 FastAPI 官方教程中的开发命令：

~~~bash
fastapi dev
~~~

启动输出中值得认识的几行大致如下：

~~~text
FastAPI Starting development server
Documentation at http://127.0.0.1:8000/docs
Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
~~~

- `Uvicorn running`：刚才亲手指挥过的 uvicorn 正在运行。
- `fastapi dev` 替我们启动了 uvicorn，开发模式等价于带上自动重载。
- 它默认寻找 `main.py`，所以连 `main:app` 也省略了。
- 正式上线时使用 `fastapi run`，不带开发时的自动重启。
- `Documentation` 行先记住，下一节会查看自动生成的接口文档。

两条命令做的是同一件事。本课程后续统一使用更简洁、也与官网一致的 `fastapi dev`；在其他教程里遇到 `uvicorn main:app --reload` 时，要知道它就是“手动挡”写法。

### 验证 GET 接口

新开一个终端调用接口：

~~~bash
curl http://localhost:8000/api/profile
~~~

返回的 JSON 和手搓版一样。

再故意访问一个不存在的路径：

~~~bash
curl http://localhost:8000/nope
~~~

FastAPI 会自动返回类似：

~~~json
{"detail":"Not Found"}
~~~

这里我们一行 404 代码都没有写，而且错误响应也带着 JSON 格式的说明，比手搓版只返回空 404 更完整。

## 目录里多了一个：`__pycache__`

接口跑通后，回到 `backend` 目录看一眼：

~~~bash
ls
ls __pycache__
~~~

Python 运行代码时会自动生成 `__pycache__/`，里面是缓存文件，可以让 Python 下次加载代码时更快。它不是源码，删掉也没关系，运行代码时还会重新生成。

它和 `.venv/`、前端的 `node_modules/` 一样，都是可以重新生成的本地文件，不应该提交到 Git。在项目的 `.gitignore` 中加入：

~~~gitignore
__pycache__/
*.py[cod]
~~~

第一行忽略所有层级的 `__pycache__` 目录，第二行忽略 `.pyc`、`.pyo`、`.pyd` 等 Python 生成文件。

## 惊喜：我们的 API 自己长出了文档

浏览器打开：

~~~text
http://localhost:8000/docs
~~~

这里会出现一个接口文档页面，并列出 `/api/profile`。展开它，依次点击 **Try it out** 和 **Execute**，页面会真正调用一次接口。

留意其中的：

- **Request URL**：请求地址。
- **Response body**：响应内容。
- **Response code**：状态码。

回想 5.1：当时我们是照着 DeepSeek 的文档学会调用 API 的，文档是 API 的说明书。现在，FastAPI 自动替我们的 API 生成了说明书，代码修改后文档也会跟着变化。

这就是框架把通用工作打包起来的另一个例子。

> 如果 `/docs` 打开是一片空白，多半是文档页面依赖的公共 CDN 网络不稳定。可以刷新几次或更换网络；接口本身不受影响。

## 加一个 POST 接口：`/api/analyze`

文字实验室需要一个接口：提交一段文字，返回分析结果。“提交内容”对应 HTTP 的 POST 方法。

真正的拼音和情感分数要到模块 6 才用第三方库实现，所以本节先把接口的**形状**立起来。

### 先把需求说清楚：一来一回长什么样

调用方提交什么？请求体只需要一个字段：

~~~json
{
  "text": "今天的风很轻"
}
~~~

服务端返回什么？文字实验室的结果卡需要四个位置：

~~~json
{
  "text": "今天的风很轻",
  "score": 0.5,
  "label": "偏平静",
  "pinyin": "(模块 6 再说)"
}
~~~

本节先把原文照抄返回，其他三个字段使用占位值。接口的数据形状先确定下来，里面的具体算法以后可以更换。这也是 API 的价值之一：**约定不变，实现可以替换。**

5.3 没有手搓 POST，是因为它需要更多底层工作：

- 读取 `Content-Length`；
- 接收请求体字节；
- 解析 JSON；
- 校验字段是否完整、类型是否正确。

现在看看 FastAPI 如何处理这些工作。

### 动手：三步写完

这个接口一共改三处。

**第一步：**在 `main.py` 的 import 区加入：

~~~python
from pydantic import BaseModel
~~~

**第二步：**在 `profile` 后面声明请求体模型：

~~~python
class AnalyzeRequest(BaseModel):
    text: str
~~~

这不是在处理请求，而是在声明：

> 这类请求的请求体必须有 `text` 字段，并且它必须是字符串。

**第三步：**在文件末尾加入接口：

~~~python
@app.post("/api/analyze")
def analyze(req: AnalyzeRequest):
    return {
        "text": req.text,
        "score": 0.5,
        "label": "偏平静",
        "pinyin": "(模块 6 再说)",
    }
~~~

这三步写完后，完整的 `main.py` 是：

~~~python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

profile = {
    "heroTitle": "关于我",
    "heroSubtitle": "项目，创意，灵感，心得，我的作品",
}


class AnalyzeRequest(BaseModel):
    text: str


@app.get("/api/profile")
def get_profile():
    return profile


@app.post("/api/analyze")
def analyze(req: AnalyzeRequest):
    return {
        "text": req.text,
        "score": 0.5,
        "label": "偏平静",
        "pinyin": "(模块 6 再说)",
    }
~~~

没写任何手动读取请求体或手动设置响应头的代码。

### 回头看：这三步在做什么

- `BaseModel` 来自 `pydantic`，是 FastAPI 的数据模型基类，专门管理数据的解析和校验。
- `class AnalyzeRequest(BaseModel)` 把请求体的形状写下来：必须有 `text`，类型是 `str`。
- `@app.post("/api/analyze")` 表示 `/api/analyze` 的 POST 请求交给下面的函数。
- 参数 `req: AnalyzeRequest` 让 FastAPI 自动读取请求体、解析 JSON、校验字段，并转换成容易使用的对象。
- 函数中直接使用 `req.text`，再返回约定好的响应形状。

| 我们写的声明 | FastAPI 自动完成 |
| --- | --- |
| `text` | 要求请求体中有这个字段 |
| `: str` | 要求这个字段是字符串 |
| `req: AnalyzeRequest` | 把 JSON 请求体解析成一个好用的对象 |
| 字段缺失或类型不对 | 返回校验错误 |

代码中的声明同时带来了自动校验和自动文档。

### 测试正确的请求

开发模式会自动重启。新开一个终端执行：

~~~bash
curl http://localhost:8000/api/analyze \
  -H "Content-Type: application/json" \
  -d '{"text": "今天的风很轻，适合把想法写下来"}'
~~~

返回结果的形状应当是：

~~~json
{
  "text": "今天的风很轻，适合把想法写下来",
  "score": 0.5,
  "label": "偏平静",
  "pinyin": "(模块 6 再说)"
}
~~~

这条命令和 5.1 调用 DeepSeek 的命令结构很像：

- `-H "Content-Type: application/json"` 声明请求体格式。
- `-d '{...}'` 提交 JSON 请求体。
- `-d` 会让 `curl` 默认使用 POST。

当时我们是调用方；现在我们自己就是 API 的提供方。

### 故意发错请求：422

把字段名故意写错，并加上 `-i`，让 curl 同时显示响应头和状态码：

~~~bash
curl -i http://localhost:8000/api/analyze \
  -H "Content-Type: application/json" \
  -d '{"txt": "字段名写错了"}'
~~~

第一行会看到：

~~~text
HTTP/1.1 422 Unprocessable Entity
~~~

响应 JSON 会指出缺少 `text` 字段。校验代码一行都没有写，是 `AnalyzeRequest` 的声明和 FastAPI 自动完成的。

再回到 `/docs` 刷新，`/api/analyze` 已经自动出现。展开它可以看到请求体必须包含一个字符串类型的 `text` 字段。

> `score`、`label`、`pinyin` 目前都是写死的占位值。本节只负责把接口的形状立住；模块 6 会把它们换成真正的拼音和情感分数。

## 最后一件事：报错了怎么看

故意制造一个服务端 bug：把 `analyze` 中的 `req.text` 改成不存在的 `req.txt`：

~~~python
return {
    "text": req.txt,  # 故意写错
    "score": 0.5,
    "label": "偏平静",
    "pinyin": "(模块 6 再说)",
}
~~~

保存后，用刚才那份正确请求再次调用，并加上 `-i`：

~~~bash
curl -i http://localhost:8000/api/analyze \
  -H "Content-Type: application/json" \
  -d '{"text": "测试"}'
~~~

这次第一行是：

~~~text
HTTP/1.1 500 Internal Server Error
~~~

`5xx` 表示服务方的问题，这次确实是我们代码的问题。

对比一下：

- 调用方把字段名写错，得到 `422`，属于请求校验错误。
- 服务端代码写错，得到 `500`，属于服务端内部错误。

服务进程通常不会因为一次请求报错就退出，修好代码后仍能继续接收请求。终端会打印一大段 traceback（错误回溯）。读报错有固定套路：

1. **先看最后一行**：例如 `AttributeError: 'AnalyzeRequest' object has no attribute 'txt'`，它告诉你错误类型和原因。
2. **再往上找自己的文件**：找到 `File ".../main.py", line XX`，它告诉你实际出错的文件和行号。行号会因空行和具体写法不同而变化。

改回 `req.text`，保存并重新请求，接口就会恢复正常。

> 报错不是需要躲开的东西，而是线索：最后一行说明“是什么错”，上面的文件行号说明“在哪儿”。实在读不懂，可以把完整 traceback 交给 AI 分析。

## 收尾：把依赖清单更新好

两个接口写完、依赖也确定后，把今天安装的依赖记录到 `requirements.txt`。

如果 5.2 的 `requirements.txt` 中原本只有 `requests`，重新生成：

~~~bash
pip freeze > requirements.txt
cat requirements.txt
~~~

Windows PowerShell 查看文件内容：

~~~powershell
pip freeze > requirements.txt
Get-Content requirements.txt
~~~

`fastapi`、`uvicorn`，以及 `starlette`、`pydantic` 等依赖都会进入清单。`[standard]` 套餐会让清单比 5.2 更长。

`.venv/` 和 `__pycache__/` 不提交 Git，但 `requirements.txt` 要和代码一起提交，其他人才能按清单重建相同的环境。

## 这一节应该带走什么

- **框架**：把每个后端都要做的通用工作打包复用；因为我们在 5.3 手搓过一次，所以知道 FastAPI 替我们省掉了什么。
- **主流框架**：Flask、Django、FastAPI 的概念相通。
- **职责分工**：FastAPI 负责定义接口，uvicorn 负责运行服务器；`uvicorn main:app --reload` 是手动启动方式，`fastapi dev` 是更简洁的开发命令。
- **安装方式**：`pip install "fastapi[standard]"` 会安装 FastAPI 和推荐配件；使用 uv 的项目也可以执行 `uv add "fastapi[standard]"`。
- **缓存文件**：Python 生成的 `__pycache__/` 可以重新生成，要写入 `.gitignore`；依赖清单用 `pip freeze` 更新。
- **自动文档**：访问 `/docs`，我们的 API 会自动获得可交互的说明书。
- **请求模型**：用 Pydantic 的 `BaseModel` 声明请求体，FastAPI 会自动解析 JSON、校验字段，并把声明同步到文档。
- **错误定位**：调用方字段错误通常是 `422`；服务端代码错误通常是 `500`。读 traceback 时先看最后一行，再找自己文件的行号。
- **接口设计**：`/api/analyze` 先固定请求和响应的形状，真正的拼音和情感分析在模块 6 再实现；约定不变，实现可以替换。

下一节，前端和后端正式握手，让网页来调用这两个接口。